# 02 — Knowledge Graph Construction & GraphSAGE Embeddings

**Project:** MARS — Multi-Agent Recommender System for Personalized Learning  
**Agent:** KnowledgeGraphAgent  
**Purpose:** Build the EdNet knowledge graph, mine prerequisite relations,
train GraphSAGE embeddings, and generate publication-quality figures.

In [ ]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from collections import defaultdict

plt.style.use("seaborn-v0_8-paper")
plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "figure.figsize": (8, 5),
    "savefig.bbox": "tight",
})

RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

print("Libraries loaded.")

## 1. Load Data & Build Graph

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s")

from data.loader import EdNetLoader
from agents.kg_agent import KnowledgeGraphAgent

# Load metadata
loader = EdNetLoader(data_dir="../data/raw")
questions = loader.load_questions()
lectures = loader.load_lectures()

# Load interactions (sample for speed)
SAMPLE_USERS = 1000
interactions = loader.load_interactions(sample_users=SAMPLE_USERS)

print(f"Questions: {len(questions):,}")
print(f"Lectures:  {len(lectures):,}")
print(f"Interactions: {len(interactions):,} ({interactions['user_id'].nunique()} users)")

In [ ]:
# Build the knowledge graph
kg_agent = KnowledgeGraphAgent()
kg_agent.build_graph(questions, lectures)

# Update question difficulties from interactions
kg_agent.update_difficulties(interactions)

stats = kg_agent.get_graph_stats()
for k, v in stats.items():
    print(f"  {k}: {v}")

## 2. Graph Statistics (Table for Paper)

In [ ]:
G = kg_agent.graph

summary_data = {
    "Entity": ["Question", "Lecture", "Tag (Skill)", "Part", "Total Nodes",
               "HAS_TAG", "COVERS_TAG", "BELONGS_TO_PART", "Total Edges"],
    "Count": [
        stats["node_types"].get("question", 0),
        stats["node_types"].get("lecture", 0),
        stats["node_types"].get("tag", 0),
        stats["node_types"].get("part", 0),
        stats["total_nodes"],
        stats["edge_types"].get("HAS_TAG", 0),
        stats["edge_types"].get("COVERS_TAG", 0),
        stats["edge_types"].get("BELONGS_TO_PART", 0),
        stats["total_edges"],
    ],
}
summary_df = pd.DataFrame(summary_data)
summary_df.style.hide(axis="index")

## 3. Mine Prerequisite Relations

In [ ]:
# Mine prerequisite edges from student mastery sequences
kg_agent.build_prerequisites(interactions)

# Updated stats
stats2 = kg_agent.get_graph_stats()
print(f"PREREQUISITE_OF edges: {stats2['edge_types'].get('PREREQUISITE_OF', 0)}")
print(f"Prerequisite DAG nodes: {stats2['prerequisite_dag_nodes']}")
print(f"Prerequisite is DAG: {stats2['prerequisite_is_dag']}")
print(f"Average degree: {stats2['avg_degree']}")

## 4. Visualise Prerequisite Graph

In [ ]:
# Extract prerequisite subgraph
prereq_edges = [
    (u, v, d)
    for u, v, d in G.edges(data=True)
    if d.get("edge_type") == "PREREQUISITE_OF"
]
prereq_sub = nx.DiGraph()
for u, v, d in prereq_edges:
    prereq_sub.add_edge(u, v, **d)

if prereq_sub.number_of_nodes() > 0:
    fig, ax = plt.subplots(figsize=(14, 10))

    # Color by part_id
    part_colors = {1: "#e74c3c", 2: "#3498db", 3: "#2ecc71", 4: "#f39c12",
                   5: "#9b59b6", 6: "#e67e22", 7: "#1abc9c"}
    node_colors = []
    labels = {}
    for n in prereq_sub.nodes():
        ndata = G.nodes[n]
        tid = ndata.get("tag_id", "?")
        labels[n] = str(tid)
        pids = ndata.get("part_ids", set())
        if isinstance(pids, set) and pids:
            pid = min(pids)
        else:
            pid = 0
        node_colors.append(part_colors.get(pid, "#95a5a6"))

    pos = nx.spring_layout(prereq_sub, k=2.0, seed=42)
    nx.draw_networkx_nodes(prereq_sub, pos, node_color=node_colors,
                           node_size=400, alpha=0.9, ax=ax)
    nx.draw_networkx_labels(prereq_sub, pos, labels, font_size=7, ax=ax)
    nx.draw_networkx_edges(prereq_sub, pos, alpha=0.5,
                           edge_color="#555", arrows=True,
                           arrowstyle="->", arrowsize=12, ax=ax)

    # Legend
    from agents.kg_agent import PART_NAMES
    patches = [mpatches.Patch(color=c, label=f"Part {p}: {PART_NAMES[p]}")
               for p, c in part_colors.items()]
    ax.legend(handles=patches, loc="upper left", fontsize=8, framealpha=0.8)
    ax.set_title(f"Tag Prerequisite Graph ({prereq_sub.number_of_nodes()} tags, "
                 f"{prereq_sub.number_of_edges()} edges)")
    ax.axis("off")

    fig.savefig(RESULTS_DIR / "fig_prerequisite_graph.png")
    plt.show()
else:
    print("No prerequisite edges found (try with more users).")

## 5. Prerequisite Chains

In [ ]:
chains = kg_agent.get_prerequisite_chains(max_chains=15)

if chains:
    print(f"Found {len(chains)} prerequisite chains (showing longest):")
    print()
    for i, chain in enumerate(chains[:10], 1):
        arrow_str = " -> ".join(str(t) for t in chain)
        print(f"  Chain {i} (len={len(chain)}): {arrow_str}")
else:
    print("No chains found (try with more users for richer prerequisite mining).")

## 6. Train GraphSAGE Embeddings

In [ ]:
embeddings = kg_agent.train_graphsage(
    hidden=128,
    out_dim=64,
    epochs=200,
    lr=0.01,
)
print(f"Tag embeddings shape: {embeddings.shape}")

## 7. t-SNE Visualisation of Tag Embeddings

In [ ]:
from sklearn.manifold import TSNE

if embeddings is not None and embeddings.shape[0] > 5:
    perplexity = min(30, embeddings.shape[0] - 1)
    tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42, n_iter=1000)
    emb_2d = tsne.fit_transform(embeddings)

    # Map each tag to its primary part
    tag_nodes_sorted = sorted(
        [n for n, d in G.nodes(data=True) if d.get("node_type") == "tag"],
        key=lambda n: G.nodes[n]["tag_id"],
    )
    primary_parts = []
    for tn in tag_nodes_sorted:
        pids = G.nodes[tn].get("part_ids", set())
        primary_parts.append(min(pids) if isinstance(pids, set) and pids else 0)

    part_colors_map = {0: "#95a5a6", 1: "#e74c3c", 2: "#3498db", 3: "#2ecc71",
                       4: "#f39c12", 5: "#9b59b6", 6: "#e67e22", 7: "#1abc9c"}
    colors = [part_colors_map.get(p, "#95a5a6") for p in primary_parts]

    fig, ax = plt.subplots(figsize=(10, 8))
    scatter = ax.scatter(emb_2d[:, 0], emb_2d[:, 1], c=colors, s=50, alpha=0.7, edgecolors="white", linewidth=0.3)

    patches = [mpatches.Patch(color=part_colors_map[p], label=f"Part {p}")
               for p in sorted(set(primary_parts)) if p > 0]
    ax.legend(handles=patches, title="Primary TOEIC Part", fontsize=9)
    ax.set_title("t-SNE of GraphSAGE Tag Embeddings (colored by TOEIC Part)")
    ax.set_xlabel("t-SNE dim 1")
    ax.set_ylabel("t-SNE dim 2")
    sns.despine()

    fig.savefig(RESULTS_DIR / "fig_tsne_tag_embeddings.png")
    plt.show()
else:
    print("Not enough embeddings for t-SNE.")

## 8. Cold-Start Demo

In [ ]:
# Simulate a diagnostic with a few question responses
sample_questions = questions.sample(10, random_state=42)
simulated_diagnostic = {
    "responses": [
        {"question_id": row["question_id"], "correct": i % 3 != 0}
        for i, (_, row) in enumerate(sample_questions.iterrows())
    ]
}

cs_result = kg_agent.handle_cold_start(
    user_id="demo_user",
    diagnostic=simulated_diagnostic,
)

print(f"Mastered tags:  {cs_result['mastered_tags']}")
print(f"Gap tags:       {cs_result['gap_tags']}")
print(f"Prerequisite gaps: {cs_result['prerequisite_gaps']}")
print(f"\nRecommendations ({cs_result['n_recommendations']}):")
for r in cs_result["recommendations"][:5]:
    print(f"  {r['item_type']:8s} {r['item_id']:8s}  priority={r['priority']:.1f}  reason: {r['reason']}")

## 9. Degree Distribution

In [ ]:
# Tag degree distribution (in the full graph)
tag_nodes = [n for n, d in G.nodes(data=True) if d.get("node_type") == "tag"]
tag_degrees = [G.degree(n) for n in tag_nodes]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(tag_degrees, bins=40, color="#3498db", edgecolor="white", linewidth=0.3)
axes[0].set_xlabel("Degree (total connections)")
axes[0].set_ylabel("Number of Tags")
axes[0].set_title("Tag Node Degree Distribution")
axes[0].axvline(np.median(tag_degrees), color="red", linestyle="--",
                label=f"Median: {np.median(tag_degrees):.0f}")
axes[0].legend()

# Tag difficulty distribution
tag_diffs = [G.nodes[n].get("avg_difficulty", 0.5) for n in tag_nodes]
axes[1].hist(tag_diffs, bins=30, color="#e74c3c", edgecolor="white", linewidth=0.3)
axes[1].set_xlabel("Average Difficulty")
axes[1].set_ylabel("Number of Tags")
axes[1].set_title("Tag Difficulty Distribution")

for ax in axes:
    sns.despine(ax=ax)

fig.tight_layout()
fig.savefig(RESULTS_DIR / "fig_tag_degree_difficulty.png")
plt.show()

In [ ]:
# Final summary
final_stats = kg_agent.get_graph_stats()
print("\n=== Knowledge Graph Summary ===")
for k, v in final_stats.items():
    print(f"  {k}: {v}")

if kg_agent.tag_embeddings is not None:
    print(f"\nGraphSAGE embeddings: {kg_agent.tag_embeddings.shape}")
    print(f"Saved to: ../models/tag_embeddings.npy")

print(f"\nFigures saved to: {RESULTS_DIR.resolve()}")